# Fine-Tuning Generation Models
## The Three LLM Training Steps: Pretraining, Supervised Fine-Tuning, and Preference Tuning

### 1. Language Modeling
- The first step in creating a high-quality LLM is to pretrain it on one or more massive text datasets
-  During training, it attempts to predict the next token to accurately learn linguistic and semantic representations found in the text. 
- this is called language modeling and is a self-supervised method.
- This produces a base model, also commonly referred to as a pretrained or foundation model.
- Base models are a key artifact of the training process but are harder for the end user to deal with.

### 2. Fine-tuning (supervised fine-tuning)

- LLMs are more useful if they respond well to instructions and try to follow them.
- When humans ask the model to write an article, they expect the model to generate the article and not list other instructions for example (which is what a base model might do).
- With supervised fine-tuning (SFT), we can adapt the base model to follow instructions.
- During this fine-tuning process, the parameters of the base model are updated to be more in line with our target task, like following instructions.
- Like a pretrained model, it is trained using next-token prediction but instead of only predicting the next token, it does so based on a user input. Check img2.
- SFT can also be used for other tasks, like classification, but is often used to go from a base generative model to an instruction (or chat) generative model.

### 3. Fine-tuning 2 (Preference tuning)

- The final step further improves the quality of the model and makes it more aligned with the expected behavior of AI safety or human preferences. 
-  This is called **preference tuning**.
-  Preference tuning is a form of fine-tuning and, as the name implies, aligns the output of the model to our preferences, which are defined by the data that we give it.
- Like SFT, it can improve upon the original model but has the added benefit of distilling preference of output in its training process.

The above three steps are illustrated in img3 and demonstrate the process of starting from an untrained architecture and ending with a preference-tuned LLM.

## Supervised Fine-Tuning (SFT)
- The purpose of pretraining a model on large datasets is that it is able to reproduce language and its meaning. During this process, the model learns to complete input phrases as shown in

### Full Fine tuning
- The most common fine-tuning process is full fine-tuning. Like pretraining an LLM, this process involves updating all parameters of a model to be in line with your target task.
-  The main difference is that we now use a smaller but labeled dataset whereas the pretraining process was done on a large dataset without any labels. Img5
- You can use any labeled data for full fine-tuning, making it also a great technique for learning domain-specific representations. 
To make our LLM follow instructions, we will need question-response data. This data, as shown in Figure 12-7, is queries by the user with corresponding answers. img6

- During full fine-tuning, the model takes the input (instructions) and applies next-token prediction on the output (response). In turn, instead of generating new questions, it will follow instructions.

### Parameter-Efficient Fine-Tuning (PEFT)
- Updating all parameters of a model has a large potential of increasing its performance but comes with several disadvantages.
- It is costly to train, has slow training times, and requires significant storage.
- To resolve these issues, attention has been given to parameter-efficient fine-tuning (PEFT) alternatives that focus on fine-tuning pretrained models at higher computational efficiency.

#### Adapters
- Adapters are a core component of many PEFT-based techniques. 
- The method proposes a set of additional modular components inside the Transformer that can be fine-tuned to improve the model’s performance on a specific task without having to fine-tune all the model weights. This saves a lot of time and compute.
- Adapters are described here: https://arxiv.org/abs/1902.00751, which showed that fine-tuning 3.6% of the parameters of BERT for a task can yield comparable performance to fine-tuning all the model’s weights.
-  On the GLUE benchmark, the authors show they reach within 0.4% of the performance of full fine-tuning. 
-  In a single Transformer block, the paper’s proposed architecture places adapters after the attention layer and the feedforward neural network as illustrated in img7
- It’s not enough to only alter one Transformer block, however, so these components are part of every block in the model. img 8

- The paper: https://arxiv.org/abs/2007.07779, introduced the Adapter Hub as a central repository for sharing adapters
- A lot of these earlier adapters were more focused on BERT architectures.
- More recently, the concept has been applied to text generation Transformers in papers like:  LLaMA-Adapter: Efficient fine-tuning of language models with zero-init attention: https://arxiv.org/abs/2303.16199.

#### Low-Rank Adaptation (LoRA)
- As an alternative to adapters, low-rank adaptation (LoRA) was introduced and is at the time of writing is a widely used and effective technique for PEFT. 
-  LoRA is a technique that (like adapters) only requires updating a small set of parameters.
-  As illustrated in Figure img10, it creates a small subset of the base model to fine-tune instead of adding layers to the model
- Like adapters, this subset allows for much quicker fine-tuning since we only need to update a small part of the base model. 
- We create this subset of parameters by approximating large matrices that accompany the original LLM with smaller matrices. 
- We can then use those smaller matrices as a replacement and fine-tune them instead of the original large matrices. Take for example the 10 × 10 matrix we see in img10.
- We can come up with two smaller matrices, which when multiplied, reconstruct a 10 × 10 matrix. This is a major efficiency win because instead of using 100 weights (10 times 10) we now only have 20 weights (10 plus 10), as we can see in img12

- During training, we only need to update these smaller matrices instead of the full weight changes. The updated change matrices (smaller matrices) are then combined with the full (frozen) weights as illustrated in img13
- Papers like “Intrinsic dimensionality explains the effectiveness of language model fine-tuning” demonstrate that language models “have a very low intrinsic dimension.”: https://arxiv.org/abs/2012.13255
- This means that we can find small ranks that approximate even the massive matrices of an LLM. 
- A 175B model like GPT-3, for example, would have a weight matrix of 12,288 × 12,288 inside each of its 96 Transformer blocks. 
That’s 150 million parameters. 
- If we can successfully adapt that matrix into rank 8, that would only require two 12,288 × 2 matrices resulting in 197K parameters per block. These are major savings in speed, storage, and compute as explained further in the previously LoRA paper: LoRA: https://arxiv.org/abs/2106.09685 .
This smaller representation is quite flexible in that you can select which parts of the base model to fine-tune. For instance, we can only fine-tune the Query and Value weight matrices in each Transformer layer.

#### Compressing the model for (more) efficient training
- We can make LoRA even more efficient by reducing the memory requirements of the model’s original weights before projecting them into smaller matrices.
- The weights of an LLM are numeric values with a given precision, which can be expressed by the number of bits like float64 or float32. As illustrated in img14, if we lower the amount of bits to represent a value, we get a less accurate result. However, if we lower the number of bits we also lower the memory requirements of that model.

- With quantization, we aim to lower the number of bits while still accurately representing the original weight values. However, as shown in Figure 15, when directly mapping higher precision values to lower precision values, multiple higher precision values might end up being represented by the same lower precision values.

- Instead, the authors of QLoRA, a quantized version of LoRA, found a way to go from a higher number of bits to a lower value and vice versa without differentiating too much from the original weights.

- They used blockwise quantization to map certain blocks of higher precision values to lower precision values. 
- Instead of directly mapping higher precision to lower precision values, additional blocks are created that allow for quantizing similar weights. 
As shown in img16, this results in values that can be accurately represented with lower precision.
- A nice property of neural networks is that their values are generally normally distributed between –1 and 1. This property allows us to bin the original weights to lower bits based on their relative density, as illustrated in img17. The mapping between weights is more efficient as it takes into account the relative frequency of weights. This also reduces issues with outliers.

- Combined with the blockwise quantization, this normalization procedure allows for accurate representation of high precision values by low precision values with only a small decrease in the performance of the LLM. 
- As a result, we can go from a 16-bit float representation to a measly 4-bit normalized float representation. 
- A 4-bit representation significantly reduces the memory requirements of the LLM during training. 
- Note that the quantization of LLMs in general is also helpful for inference as quantized LLMs are smaller in size and therefore require less VRAM.
- Linke toread more about QLora and quanitization:
   -QLoRA: Efficient Finetuning of Quantized LLMs: https://arxiv.org/abs/2305.14314
   - A Visual Guide to Quantization: https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-quantization

## Instruction tuning with QLoRa
- Now that we have explored how QLoRA works, let us put that knowledge into practice! 
- In this section, we will fine-tune a completely open source and smaller version of Llama, TinyLlama: https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T to follow instructions using the QLoRA procedure. 
- Consider this model a base or pretrained model, one that was trained with language modeling but cannot yet follow instructions.

### Step1: Check Template Instuction Data
- To have the LLM follow instructions, we will need to prepare instruction data that follows a chat template.
-  This chat template, as illustrated in Figure 18, differentiates between what the LLM has generated and what the user has generated.
- We chose this chat template to use throughout the examples since the chat version of TinyLlama uses the same format. The data that we are using is a small subset of the UltraChat dataset. 
- This dataset is a filtered version of the original UltraChat dataset that contains almost 200k conversations between a user and an LLM.
- Link to the dataset: https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k


In [ ]:
pip install -q accelerate==0.31.0 peft==0.11.1 bitsandbytes>=0.43.2 transformers==4.41.2 trl==0.9.4 sentencepiece==0.2.0

## Supervised Fine-Tuning (SFT)
### Step1: Data Processing

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load the tokenizer
template_tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)




In [ ]:
import torch
import sys

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLlama is using"""
    # Format answer
    chat= example["messages"]
    prompt= template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}


In [ ]:
# Load and format the data using the template TinyLLama is using
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k",  split="test_sft")
      .shuffle(seed=42)
      .select(range(3_000))
)
dataset

In [ ]:
dataset = dataset.map(format_prompt)

In [ ]:
dataset

In [ ]:
print(dataset.column_names)

We select a subset of 3,000 documents to reduce the training time, but you can increase this value to get more accurate result

In [ ]:
# Example of formatted prompt
print(dataset["text"][2576])

### Step2: Model Quantization
- Now that we have our data, we can start loading in our model. This is where we apply the Q in QLoRA, namely quantization. We use the bitsandbytes package to compress the pretrained model to a 4-bit representation.
- In BitsAndBytesConfig, you can define the quantization scheme. We follow the steps used in the original QLoRA paper and load the model in 4-bit (load_in_4bit) with a normalized float representation (bnb_4bit_quant_type) and double quantization (bnb_4bit_use_double_quant):

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-bits quantization configuration

bnb_config= BitsAndBytesConfig(
    load_in_4bit=True,# Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",# Use NF4 quantization
    bnb_4bit_use_double_quant=True,# Apply nested quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute dtype
)

# Load the model to train on GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # Leave this out for regular SFT
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMa tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

In [ ]:
import torch
import bitsandbytes as bnb
import transformers
import accelerate

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("bitsandbytes:", bnb.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

This quantization procedure allows us to decrease the size of the original model while retaining most of the original weights’ precision. Loading the model now only uses ~1 GB VRAM compared to the ~4 GB of VRAM it would need without quantization. Note that during fine-tuning, more VRAM will be necessary so it does not cap out on the ~1 GB VRAM needed to load the model.

### Configuration
#### LoRa configuration
Next, we will need to define our LoRA configuration using the peft library, which represents hyperparameters of the fine-tuning process:
- Link: https://github.com/huggingface/peft

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

#Prepare LoRa configuration
peft_config= LoraConfig(
    lora_alpha=32, # LoRA scaling
    lora_dropout=0.1, # Dropout
    r=64, # Rank
    bias="none", # Bias
    task_type="CAUSAL_LM", # Task type
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# Prepare the model
# prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

There are several parameters worth mentioning:
- r: This is the rank of the compressed matrices (recall this from Figure 12-13) Increasing this value will also increase the sizes of compressed matrices leading to less compression and thereby improved representative power. Values typically range between 4 and 64.
- lora_alpha: Controls the amount of change that is added to the original weights. In essence, it balances the knowledge of the original model with that of the new task. A rule of thumb is to choose a value twice the size of r.
- target_modules: Controls which layers to target. The LoRA procedure can choose to ignore specific layers, like specific projection layers. This can speed up training but reduce performance and vice versa.

Playing around with the parameters is a worthwhile experiment to get an intuitive understanding of values that work and those that do not. You can find an amazing resource of additional tips on LoRA fine-tuning in the Ahead of AI newsletter by Sebastian Raschka. Link: https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms

- **Note**: This example demonstrates an efficient form of fine-tuning your model. If you want to perform full fine-tuning instead, you can remove the quantization_config parameter when loading the model and skip the creation of peft_config. By removing those, we would go from “Instruction tuning with QLoRA” to “full instruction tuning.”


### Training Configutaion




In [ ]:
from transformers import TrainingArguments

output_dir = "./results"

# Training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True
)



There are several parameters worth mentioning:

- num_train_epochs: The total number of training rounds. Higher values tend to degrade performance so we generally like to keep this low.
- learning_rate: Determines the step size at each iteration of weight updates. The authors of QLoRA found that higher learning rates work better for larger models (>33B parameters).
- lr_scheduler_type: A cosine-based scheduler to adjust the learning rate dynamically. It will linearly increase the learning rate, starting from zero, until it reaches the set value. After that, the learning rate is decayed following the values of a cosine function.
- optim: The paged optimizers used in the original QLoRA paper. 

Optimizing these parameters is a difficult task and there are no set guidelines for doing so. It requires experimentation to figure out what works best for specific datasets, model sizes, and target tasks.

**Note**: Although this section describes instruction tuning, we could also use QLoRA to fine-tune an instruction model. For instance, we could fine-tune a chat model to generate specific SQL code or to create JSON output that adheres to a specific format. As long as you have the data available (with appropriate query-response items), QLoRA is a great technique for nudging an existing chat model to be more appropriate for your use case.

### Training

During training the loss will be printed every 10 steps according to the logging_steps parameter. If you are using the free GPU provided by Google Colab, which is the Tesla T4 at the time of writing, then training might take up to an hour. A good time to take a break!

In [ ]:
import trl, transformers, accelerate, huggingface_hub

print("trl:", trl.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

In [ ]:
from trl import  SFTConfig

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model.config.pad_token_id = tokenizer.pad_token_id

sft_config = SFTConfig(
    output_dir="./results",
    dataset_text_field="text",   # important
    max_seq_length=512,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,

    save_strategy="no",
    report_to="none",
)

In [ ]:
from trl import SFTTrainer

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    # Leave this out for regular SFT
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer,
)

# Train model
trainer.train()

In [ ]:
from trl import SFTTrainer, SFTConfig

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
model.config.pad_token_id = tokenizer.pad_token_id

sft_config = SFTConfig(
    output_dir="./results",
    dataset_text_field="text",
    max_length=512,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,

    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
print(dataset)
print(dataset.column_names)
print(dataset[0])

In [ ]:
trainer.train()

In [ ]:
# Save QLoRA weights
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

After we have trained our QLoRA weights, we still need to combine them with the original weights to use them. We reload the model in 16 bits, instead of the quantized 4 bits, to merge the weights. Although the tokenizer was not updated during training, we save it to the same folder as the model for easier access:

In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

## Evaluating Generative Models
### Word Level Metrics
- One common metrics category for comparing generative models is word-level evaluation. 
- These classic techniques compare a reference dataset with the generated tokens on a token(set) level. Common word-level metrics include perplexity, ROUGE, BLEU, and BERTScore.
### Benchmarks
- A common method for evaluating generative models on language generation and understanding tasks is on well-known and public benchmarks, such as MMLU, GLUE, TruthfulQA, GSM8k, and HellaSwag.
- These benchmarks give us information about basic language understanding but also complex analytical answering, like math problems.
### LeaderBoards
- With so many different benchmarks, it is hard to choose which benchmark best suits your model. Whenever a model is released, you will often see it evaluated on several benchmarks to showcase how it performs across the board.
- open LLM leaderboard: https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard

### Automated Evaluation
- Part of evaluating a generative output is the quality of its text. 
- For instance, even if two models were to give the same correct answer to a question, the way they derived that answer might be different. 
- It is often not just about the final answer but also the construction of it. Similarly, although two summaries might be similar, one could be significantly shorter than another, which is often important for a good summary.
- To evaluate the quality of the generated text above the correctness of the final answer, LLM-as-a-judge was introduced. 
- In essence, a separate LLM is asked to judge the quality of the LLM to be evaluated. 
- An interesting variant of this method is **pairwise comparison**. Two different LLMs will generate an answer to a question and a third LLM will be the judge to declare which is better.
- As a result, this methodology allows for automated evaluation of open-ended questions. 
- A major advantage is that as LLMs improve, so do their capabilities to judge the quality of output. In other words, this evaluation methodology grows with the field.
### Human Evaluation
- Although benchmarks are important, the gold standard of evaluation is generally considered to be human evaluation. 
- Even if an LLM scores well on broad benchmarks, it still might not score well on domain-specific tasks. Moreover, benchmarks do not fully capture human preference and all methods discussed before are merely proxies for that.
- A great example of a human-based evaluation technique is the **Chatbot Arena**.
- When you go to this leaderboard you are shown two (anonymous) LLMs you can interact with. 
- Any question or prompt you ask will be sent to both models and you will receive their output. 
- Then, you can decide which output you prefer. This process allows for the community to vote on which models they prefer without knowing which ones are presented. Only after you vote do you see which model generated which text.
- At the time of writing, this method has generated over 800,000+ human votes that were used to compute a leaderboard. 
- These votes are used to calculate the relative skill level of LLMs based on their win rates. For instance, if a low-ranked LLM beats a high-ranked LLM, its ranking changes significantly. In chess, this is referred to as the **Elo rating system**.
- This methodology therefore uses crowdsourced votes, which helps us understand the quality of the LLM. However, it is still the aggregated opinion of a wide variety of users, which might not relate to your use case.
- As a result, there is no one perfect method of evaluating LLMs. All mentioned methodologies and benchmarks provide an important, although limited evaluation perspective. Our advice is to evaluate your LLM based on the intended use case. For coding, HumanEval would be more logical than GSM8k.
- But most importantly, we believe that you are the best evaluator. Human evaluation remains the gold standard because it is up to you to decide whether the LLM works for your intended use case. 
- As with the examples in this chapter, we highly advise that you also try these models and perhaps develop some questions yourself. For example, the authors of this book are Arabic (Jay Alammar) and Dutch (Maarten Grootendorst), and we often ask questions in our native language when approached with new models.

- In the context of LLMs, when using a specific benchmark, we tend to optimize for that benchmark regardless of the consequences. 
- For instance, if we focus purely on optimizing for generating grammatically correct sentences, the model could learn to only output one sentence: “This is a sentence.” 
- It is grammatically correct but tells you nothing about its language understanding capabilities. Thus, the model may excel at a specific benchmark but potentially at the expense of other useful capabilities.

## Preference-Tuning / Alignment / RLHF
- Although our model can now follow instructions, we can further improve its behavior by a final training phase that aligns it to how we expect it to behave in different scenarios.
- We can ask a person (prefernce evaluator) to evaluate the qulity of that model generation. Img 21
- Img 22, shows a preference tuning step updating the model based on that score:
  - If the score is high, the model is updated to encourage it to generate more like this type of generation
  - If the score is low, the model is updated to discourage such generations

### Automating Preference Evaluation Using Reward Models
- To automate preference evaluation, we need a step before the preference-tuning step, namely to train a reward model, as shown in Img23
- Img24 shows that to create a reward model, we take a copy of the instuction-tuned model and slightly change it so that instead of generating text, it now outputs a single score.

### The Inputs and Outputs of a Reward Model

- The way we expect this reward model to work is that we give it a prompt and a generation, and it outputs a single number indicating the preference/quality of that generation in response to that prompt. Figure25 shows the reward model generating this single number.

### Traing a Reward Model
- We cannot directly use the reward model. It needs to first be trained to properly score generations. So let’s get a preference dataset that the model can learn from.

#### Reward model training dataset
- One common shape for preference datasets is for training example to have a prompt, with one accepted generation and one rejected generation.
    - Nuance: It's not always a good versus bad generation, It can be that the two are good, but that one is better than the other.
- Img26 shows an example preference training set with two training examples
-  One way to generate preference data is to present a prompt to the LLM and have it generate two different generations. As shown in Img27, we can ask human labelers which of the two they prefer.

### Reward Model training step
- Now that we have the preference training dataset, we can proceed to train the reward model. A simplest step is that we use the reward model to:
 - Score the accepted generation.
 - Score the rejected generation.

- Img28  shows the training objective : to ensure the accepted generation has a higher score than rejected generation
- When we combine everything together as shown in Figure 29, we get the three stages to preference tuning:
  - Collect data
  - Train a rward model
  - Use the reward model to fine-tune the LLM (operating as the preference evaluator)

- Reward models are an excellent idea that can be further extended and developed. Llama 2, for example, trains two reward models: one that scores helpfulness and another that scores safety. Check img30
- A common method to fine-tune the LLM with the trained reward model is **Proximal Policy Optimization (PPO)**. 
- PPO is a popular reinforcement technique that optimizes the instruction-tuned LLM by making sure that the LLM does not deviate too much from the expected rewards.
- It was even used to train the original ChatGPT released in November 2022.

### Training No Reward Model
- A disadvantage of PPO is that it is a complex method that needs to train at least two models, the reward model and the LLM, which can be more costly than perhaps necessary.
- **Direct Preference Optimization (DPO)** is an alternative to PPO and does away with the reinforcement-based learning procedure.
- Instead of using the reward model to judge the quality of a generation, we let the LLM itself do that. 
- As illustrated in Img31, we use a copy of the LLM as the reference model to judge the shift between the reference and trainable model in the quality of the accepted generation and rejected generation.

- By calculating this shift during training, we can optimize the likelihood of accepted generations over rejected generations by tracking the difference in the reference model and the trainable model.
- To calculate this shift and its related scores, the log probabilities of the rejected generations and accepted generations are extracted from both models. 
- As illustrated in Figure32, this process is performed at a token level where the probabilities are combined to calculate the shift between the reference and trainable models.
- Using these scores, we can optimize the parameters of the trainable model to be more confident of generating the accepted generations and less confident of generating the rejected generations. 
- Compared to PPO, the authors found DPO to be more stable during training and more accurate. Due to its stability, we will be using it as our primary model for preference tuning our previously instruction-tuned model.
### Preference Tuning With DPO
#### Data Processing: Templating Alignment Data


In [8]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [9]:
from datasets import load_dataset

def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""

    # Format answers
    system = "<|system|>\n" + example['system'] + "</s>\n"
    prompt = "<|user|>\n" + example['input'] + "</s>\n<|assistant|>\n"
    chosen = example['chosen'] + "</s>\n"
    rejected = example['rejected'] + "</s>\n"

    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# Apply formatting to the dataset and select relatively short answers
dpo_dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
dpo_dataset = dpo_dataset.filter(
    lambda r:
        r["status"] != "tie" and
        r["chosen_score"] >= 8 and
        not r["in_gsm8k_train"]
)
dpo_dataset = dpo_dataset.map(format_prompt, remove_columns=dpo_dataset.column_names)
dpo_dataset

Dataset({
    features: ['chosen', 'rejected', 'prompt'],
    num_rows: 5922
})

### Model Quanitization

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig, AutoTokenizer

# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",  # Quantization type
    bnb_4bit_compute_dtype="float16",  # Compute dtype
    bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

# Merge LoRA and base model
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
    quantization_config=bnb_config,
)
merged_model = model.merge_and_unload()

# Load LLaMA tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

### Configuration
Next, we use the same LoRA configuration as before to perform the DPO training:

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare LoRA Configuration
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA Scaling
    lora_dropout=0.1,  # Dropout for LoRA Layers
    r=64,  # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=  # Layers to target
     ['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)


### Training Configuration
- For the sake of simplicity, we will use the same training arguments as we did before with one difference. 
- Instead of running for a single epoch (which can take up to two hours), we run for 200 steps instead for illustration purposes
- Moreover, we added the warmup_ratio parameter, which increases the learning rate from 0 to the learning_rate value we set for the first 10% of steps. 
- By maintaining a small learning rate at the start (i.e., warmup period), we allow the model to adjust to the data before
applying larger learning rates, therefore avoiding harmful divergence:

In [ ]:
from trl import DPOConfig

output_dir = "./results"

# Training arguments
training_arguments = DPOConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_steps=200,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    warmup_ratio=0.1
)



### Training
Now that we have prepared all our models and parameters, we can start fine-tuning our model:


In [ ]:
from trl import DPOTrainer

# Create DPO trainer
dpo_trainer = DPOTrainer(
    model,
    args=training_arguments,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    beta=0.1,
    max_prompt_length=512,
    max_length=512,
)

# Fine-tune model with DPO
dpo_trainer.train()

# Save adapter
dpo_trainer.model.save_pretrained("TinyLlama-1.1B-dpo-qlora")

We have created a second adapter. To merge both adapters, we iteratively merge the adapters with the base model:

In [ ]:
from peft import PeftModel

# Merge LoRA and base model
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)
sft_model = model.merge_and_unload()

# Merge DPO LoRA and SFT model
dpo_model = PeftModel.from_pretrained(
    sft_model,
    "TinyLlama-1.1B-dpo-qlora",
    device_map="auto",
)
dpo_model = dpo_model.merge_and_unload()

### Inference


In [ ]:
from transformers import pipeline

# Use our predefined prompt template
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=dpo_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])